## Rules-Based Approach

1. Look for rise of at least 1000 over baseline in the first two channels that occurs within 2/10 of a second.
2. Baseline is determined by a moving 5? second average
3. The rise should be in 1 before 2.
4. It should return to baseline +- 500? within 2 seconds?


**Goal:** See how well the rules-based approach works on the predictions to cut down on false positives from the predictions from the original model.

In [1]:
import pandas as pd
import numpy as np
import glob
from tqdm.notebook import tqdm
import tensorflow as tf
import pickle
import os

/home/michael/anaconda3/envs/keras/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/michael/anaconda3/envs/keras/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/michael/anaconda3/envs/keras/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/michael/anaconda3/envs/keras/lib/python3

In [2]:
epochs = 50
model = tf.keras.models.load_model(f'../models/2025/2025_12_01_model_epoch_{epochs}')

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


In [3]:
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
dateformat = DateFormatter(fmt = '%H:%M:%S:%f')

def plot_event(idx, shift = 0):
    duration = 20
    
    readings_sub = (
        readings
        .loc[idx - shift:idx - shift + duration - 1]
        [['Channel_' + str(i) for i in range(1,7)]]
        .values
    )

    fig, ax = plt.subplots(figsize = (12,6))
    times = readings.loc[idx - shift:idx - shift + duration - 1, 'Time']

    for i in range(6):
        if i == 0:
            color = 'blue'
        elif i == 1:
            color = 'red'
        elif i == 4:
            color = 'yellow'
        elif i == 5:
            color = 'black'
        else:
            color = 'grey'
        
        plt.plot(times, readings_sub[:, i], label = 'Channel_' + str(i + 1), color=color)

    ax.xaxis.set_major_formatter(dateformat)

    plt.legend(bbox_to_anchor = (1, 0.5), loc = 'center left')
    plt.ylim(-500, 10500)

In [4]:
patient_num = '01' 
new = True

filepath = f"../pickles_readings/{patient_num}_{new}.pickle"

with open(filepath, "rb") as fi:
    readings, X_test, test_indices = pickle.load(fi)
    
readings['sgb'] = readings['sgb'].replace('False', False)

In [ ]:
predictions = model.predict(
            np.array(X_test).reshape(
                len(X_test), 
                X_test[0].shape[0], 
                6, 
                1
            )
        )

In [ ]:
threshold = 0.99

prediction_indices = [test_indices[x] for x in np.where(predictions.flatten() > threshold)[0]]

true_indices = readings[readings['sgb'] == True].index.tolist()

true_positives = []
false_positives = []
for idx in tqdm(prediction_indices):
    fp = True
    for i in range(15):
        if idx - i in true_indices:
            true_positives.append(idx - i)
            fp = False
            continue
        if idx + i in true_indices:
            true_positives.append(idx + i)
            fp = False
            continue
    if fp:
        false_positives.append(idx)

true_positives = set(true_positives)

false_positives = pd.DataFrame({'fp_index': false_positives})
false_positives['diff'] = false_positives['fp_index'].diff()
false_positives = false_positives[~(false_positives['diff'].isin([1, 2, 3, 4, 5]))]

false_negatives = [x for x in true_indices if x not in true_positives]

In [ ]:
true_positives = pd.DataFrame({'index': list(true_positives)})
true_positives

In [ ]:
window_size = 10  # one unit = 1/10 of a second

In [ ]:
for channel_num in [1, 2, 5, 6]:
    readings[f'Channel_{channel_num}_prior_avg'] = readings[f'Channel_{channel_num}'].rolling(window = window_size).mean()

In [ ]:
readings = readings.iloc[::-1]

for channel_num in [1, 2, 5, 6]:
    readings[f'Channel_{channel_num}_next_avg'] = readings[f'Channel_{channel_num}'].rolling(window = window_size).mean()

readings = readings.iloc[::-1]

In [ ]:
readings.head(2)

In [ ]:
for channel_num in [1, 2, 5, 6]:
    readings[f'Channel_{channel_num}_baseline_prior'] = readings[f'Channel_{channel_num}_prior_avg'].shift(2)

In [ ]:
rise_threshold = 1000

for channel_num in [1, 2]:
    readings[f'Channel_{channel_num}_rise'] = readings[f'Channel_{channel_num}'] >= readings[f'Channel_{channel_num}_baseline_prior'] + rise_threshold

In [ ]:
## Check for a drop of 1500 in channel 1 or 2 within 2 seconds

readings = readings.iloc[::-1]

drop_threshold = 1000
window_size = 20

for channel_num in [1, 2]:
    readings[f'Channel_{channel_num}_drop'] = (readings[f'Channel_{channel_num}'] - readings[f'Channel_{channel_num}'].rolling(window_size).min()) >= drop_threshold
    
readings = readings.iloc[::-1]

In [ ]:
# Check for a valid rise and then fall in either channel 1 or channel 2
readings['Channel_1_or_2_pair'] = (((readings['Channel_1_rise']) & (readings['Channel_1_drop'])) | \
((readings['Channel_2_rise']) & (readings['Channel_2_drop'])))

In [ ]:
## Check for rise in channel 5 or 6
rise_threshold = 1000

for channel_num in [5,6]:
    readings[f'Channel_{channel_num}_rise'] = readings[f'Channel_{channel_num}'] >= readings[f'Channel_{channel_num}_baseline_prior'] + rise_threshold

In [ ]:
## Check for a drop of 1000 in channel 5 or 6 within 2 seconds

readings = readings.iloc[::-1]

drop_threshold = 1000
window_size = 20

for channel_num in [5, 6]:
    readings[f'Channel_{channel_num}_drop'] = (readings[f'Channel_{channel_num}'] - readings[f'Channel_{channel_num}'].rolling(window_size).min()) >= drop_threshold
    
readings = readings.iloc[::-1]

In [ ]:
# Check for a valid rise and then fall in either channel 1 or channel 2
readings['Channel_5_or_6_pair'] = (((readings['Channel_5_rise']) & (readings['Channel_5_drop'])) | \
((readings['Channel_6_rise']) & (readings['Channel_6_drop'])))

In [ ]:
## Look for a rise in Channel 5 or 6 soon after
readings = readings.iloc[::-1]

window_size = 3
readings[f'Channel_5_or_6_rise_soon'] = pd.concat([(readings[f'Channel_5_rise'].rolling(window = window_size).max()), 
       (readings[f'Channel_6_rise'].rolling(window = window_size).max())], axis = 1).max(axis=1).astype(bool)

readings = readings.iloc[::-1]

In [ ]:
## Also check that the rise in channel 5 is after the rise in channel 1
window_size = 5
readings['Channel_1_before_5_or_6'] = (readings['Channel_1_or_2_pair']) & (readings['Channel_5_or_6_rise_soon']) & \
~(pd.concat([(readings[f'Channel_5_rise'].rolling(window = window_size, closed='left').max()), 
       (readings[f'Channel_6_rise'].rolling(window = window_size, closed='left').max())], axis = 1).max(axis=1).astype(bool))

#### Alternative: Instead of looking in channel 5 or 6, look for a pair

In [ ]:
## Look for a rise in Channel 5 or 6 soon after
readings = readings.iloc[::-1]

window_size = 3
readings[f'Channel_5_or_6_rise_soon'] = (readings[f'Channel_5_or_6_pair'].rolling(window = window_size).max()).astype(bool)

readings = readings.iloc[::-1]

In [ ]:
## Also check that the rise in channel 5 is after the rise in channel 1
window_size = 5
readings['Channel_1_before_5_or_6'] = (readings['Channel_1_or_2_pair']) & (readings['Channel_5_or_6_rise_soon']) & \
~(readings[f'Channel_5_or_6_pair'].rolling(window = window_size, closed='left').max()).astype(bool) 

### Now, find where the drops take place and check that the drop in 5 or 6 happens before that of 1 or 2

In [ ]:
readings['Channel_1_or_2_drop'] = (readings['Channel_1_drop']) | (readings['Channel_2_drop'])
readings['Channel_5_or_6_drop'] = (readings['Channel_5_drop']) | (readings['Channel_6_drop'])

In [ ]:
### Check that the drop in 5 or 6 happens before that of 1 or 2
window_size = 5
readings['Channel_5_or_6_drop_before_1_or_2'] = (readings['Channel_5_or_6_drop']) & \
~(readings[f'Channel_1_or_2_drop'].rolling(window = window_size, closed='left').max()).astype(bool)

In [ ]:
window_size = 20
readings['Channel_5_or_6_drop_before_soon'] = readings['Channel_5_or_6_drop_before_1_or_2'].rolling(window=window_size).max().astype(bool)

### This is not working. Perhaps find when the max of channel 1/2 occurs and when the max of 5/6 occurs (over the next 2 seconds) and make sure that the time for 5/6 is before that of 1/2?

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

In [ ]:
c1 = readings[['Channel_1', 'Channel_2']].max(axis=1).to_numpy()
c2 = readings[['Channel_5', 'Channel_6']].max(axis=1).to_numpy()

W = 20

In [ ]:
c1_win = sliding_window_view(c1, W)
c2_win = sliding_window_view(c2, W)

In [ ]:
c1_max = c1_win.max(axis = 1)
c2_max = c2_win.max(axis = 1)

c1_last_pos = (c1_win == c1_max[:,None]).cumsum(axis=1).argmax(axis=1)
c2_last_pos = (c2_win == c2_max[:,None]).cumsum(axis=1).argmax(axis=1)

In [ ]:
readings['c1_last_max_idx'] = np.concatenate([c1_last_pos, np.zeros(window_size -1)])
readings['c2_last_max_idx'] = np.concatenate([c2_last_pos, np.zeros(window_size -1)])

In [ ]:
readings['Channel_5_or_6_drop_before'] = readings['c2_last_max_idx'] < readings['c1_last_max_idx']

#### The next section is not being used right now

In [ ]:
lookahead_size = 20 # How far ahead to look for a return to baseline

In [ ]:
# Option 1: Look for the rolling average to return to close to baseline

for channel_num in [1, 2]:
    readings[f'Channel_{channel_num}_lookahead'] = readings[f'Channel_{channel_num}_next_avg'].shift(-lookahead_size)
    readings[f'Channel_{channel_num}_return_to_baseline'] = np.abs(readings[f'Channel_{channel_num}_lookahead'] - readings[f'Channel_{channel_num}_baseline_prior']) <= 500

In [ ]:
# Option 2: Look for the raw value to return to close to baseline

for channel_num in [1, 2]:
    readings[f'Channel_{channel_num}_lookahead'] = readings[f'Channel_{channel_num}'].shift(-lookahead_size)
    readings[f'Channel_{channel_num}_return_to_baseline'] = np.abs(readings[f'Channel_{channel_num}'].shift(-lookahead_size) - readings[f'Channel_{channel_num}_baseline_prior']) <= 500

In [ ]:
# Option 3: Look for the any raw value to return to close to baseline in the next second

lookahead_size = 21
baseline_diff = 1500

for channel_num in [1, 2]:
    readings = readings.iloc[::-1]
    readings[f'Channel_{channel_num}_lookahead'] = readings[f'Channel_{channel_num}'].rolling(window=lookahead_size).min()
    readings = readings.iloc[::-1]

    #raw[f'Channel_{channel_num}_return_to_baseline'] = np.abs(raw[f'Channel_{channel_num}_lookahead'] - raw[f'Channel_{channel_num}_baseline_prior']) <= baseline_diff
    readings[f'Channel_{channel_num}_return_to_baseline'] = (readings[f'Channel_{channel_num}_lookahead'] - readings[f'Channel_{channel_num}_baseline_prior']) <= baseline_diff

In [ ]:
## Look for a rise in Channel 2 soon after
readings = readings.iloc[::-1]

window_size = 3
readings[f'Channel_2_rise_soon'] = readings[f'Channel_2_rise'].rolling(window = window_size).max().astype(bool)

readings = readings.iloc[::-1]

In [ ]:
# This just checks that channel 1 rose and that channel 2 rises soon after
readings['Channel_1_first'] = (readings['Channel_1_rise']) & (readings['Channel_2_rise_soon']) #& (~raw['Channel_2_rise'])

In [ ]:
# Alternative: Also check that the rise in channel 2 doesn't happen before channel 1
# Note: This may filter out too many things

window_size = 5
readings['Channel_1_first_alt'] = (readings['Channel_1_rise']) & (readings['Channel_2_rise_soon']) & (~readings['Channel_2_rise'].rolling(window = window_size, closed='left').max().astype(bool))

In [ ]:
## Check that channel 1 stays above channel 2 for the next second within a tolerance level
## Might be too strict

readings = readings.iloc[::-1]

window_size = 10
tolerance = 200
readings['Channel_1_greater'] = (
    (readings['Channel_1'] > (readings['Channel_2'] - tolerance))
    .rolling(window=window_size)
    .min()
    .astype(bool)
)

readings = readings.iloc[::-1]

In [ ]:
### Check for messiness
window_size = 10
max_mess = 20000

for channel in range(1,7):
    readings[f'Channel_{channel}_messy_check'] = (np.abs(readings[f'Channel_{channel}'].diff()).rolling(window = window_size, center=True).sum() < max_mess)
    
readings['mess_check'] = (readings['Channel_1_messy_check']) & (readings['Channel_2_messy_check']) &(readings['Channel_3_messy_check']) &(readings['Channel_4_messy_check']) &(readings['Channel_5_messy_check']) &(readings['Channel_6_messy_check'])

In [ ]:
potential_idx = (
    readings
    #.loc[readings['Channel_1_rise']]
    #.loc[readings['Channel_1_return_to_baseline']]
    #.loc[readings['Channel_2_return_to_baseline']]
    #.loc[readings['Channel_1_first']]
    #.loc[readings['Channel_1_drop']]
    #.loc[readings['Channel_2_drop']]
    .loc[readings['Channel_1_before_5_or_6']]
    .loc[readings['Channel_5_or_6_drop_before']]
    .loc[readings['mess_check']]
    .index
)

print(len(potential_idx))

In [ ]:
readings = readings.drop(columns = 'flag', errors='ignore')

readings.loc[potential_idx, 'flag'] = True
readings['flag'] = readings['flag'].fillna(False)

In [ ]:
readings['sgb'].value_counts()

In [ ]:
readings['flag'].value_counts()

Need to check for flag in next second(?)

In [ ]:
## Look for a flag soon after
readings = readings.iloc[::-1]

window_size = 15
readings[f'flag_soon'] = readings[f'flag'].rolling(window = window_size).max().astype(bool)

readings = readings.iloc[::-1]

In [ ]:
total_events = readings["sgb"].sum()

print(f'Total Events: {total_events}')

pre_filter_tp = len(true_positives)

print(f'Pre-filter true positives: {pre_filter_tp}')

true_positives_filtered = pd.merge(
    left = true_positives,
    right = readings['flag_soon'].reset_index()
)

post_filter_tp = true_positives_filtered['flag_soon'].sum()
print(f'Post-filter true positives: {post_filter_tp}')


print(f'Sensitivity: {round(post_filter_tp / total_events,3)}')

In [ ]:
false_positives_filtered = pd.merge(
    left = false_positives,
    right = readings['flag_soon'].reset_index().rename(columns = {'index': 'fp_index'})
)

print(f'Pre-Filter False Positive Count: {len(false_positives)}')

false_positive_count = false_positives_filtered['flag_soon'].sum()

print(f'Post-Filter False Positive Count: {false_positive_count}')

In [ ]:
true_positives_filtered[~true_positives_filtered['flag_soon']]

In [ ]:
idx = 364054
readings.loc[idx:idx+14, ['Time',
                          #'Channel_1',
                          #'Channel_2',
                          'Channel_1_rise',
                          'Channel_2_rise',
                          'Channel_5_rise',
                          'Channel_6_rise',
                           'Channel_1_drop',
                           'Channel_2_drop',
#                           'Channel_5_drop',
#                           'Channel_6_drop',
                          'Channel_1_or_2_pair',
                          'Channel_5_or_6_pair',
                          'Channel_1_before_5_or_6',
                          'Channel_5_or_6_drop_before',
                          'mess_check'
                          
                          #'Channel_1_return_to_baseline', 
                          #'Channel_2_return_to_baseline',
                          #'Channel_2_rise_soon',
                          #'Channel_1_first',
                          #'Channel_5_or_6_rise_soon',
#                           'Channel_1_before_5_or_6'
                          #'Channel_1_greater',
                         #'flag'
                         ]]


In [ ]:
plot_event(idx)

In [ ]:
false_positives_filtered[false_positives_filtered['flag_soon']]

In [ ]:
idx = 123493
plot_event(idx)

In [ ]:
readings.loc[idx:idx+14, ['Time', 
                          'Channel_1',
                          'Channel_1_rise',
                          'Channel_2_rise',
                             'Channel_1_drop', 
                             'Channel_2_drop',
                             'Channel_5_rise',
                             'Channel_6_rise',
                          #'Channel_1_return_to_baseline', 
                          #'Channel_2_return_to_baseline',
                          #'Channel_2_rise_soon',
                          #'Channel_1_first',
                         'flag']]


In [ ]:
window_size = 5

readings['sgb_soon_before'] = readings['sgb'].rolling(window=window_size).max().astype(bool)

readings = readings.iloc[::-1]

readings[f'sgb_soon_after'] = readings['sgb'].rolling(window=window_size).max().astype(bool)

readings = readings.iloc[::-1]

readings['sgb_near'] = (readings['sgb_soon_before']) | (readings['sgb_soon_after'])



window_size = 20

readings['flag_soon_before'] = readings['flag'].rolling(window=window_size).max().astype(bool)

readings = readings.iloc[::-1]

readings[f'flag_soon_after'] = readings['flag'].rolling(window=window_size).max().astype(bool)

readings = readings.iloc[::-1]

readings['flag_near'] = (readings['flag_soon_before']) | (readings['flag_soon_after'])

In [ ]:
readings['sgb'].sum()

In [ ]:
missed = readings[(readings['sgb']) & (~readings['flag_near'])].index.tolist()
len(missed)

In [ ]:
fp = readings[(readings['flag']) & (~readings['sgb_near'])].index.tolist()
len(fp)

In [ ]:
readings['flag'].sum()

In [ ]:
prediction_indices = readings[readings['flag']].index.tolist()

true_indices = readings[readings['sgb'] == True].index.tolist()

true_positives = []
false_positives = []
for idx in tqdm(prediction_indices):
    fp = True
    for i in range(15):
        if idx - i in true_indices:
            true_positives.append(idx - i)
            fp = False
            continue
        if idx + i in true_indices:
            true_positives.append(idx + i)
            fp = False
            continue
    if fp:
        false_positives.append(idx)
        
true_positives = set(true_positives)

false_positives = pd.DataFrame({'fp_index': false_positives})
false_positives['diff'] = false_positives['fp_index'].diff()
false_positives = false_positives[~(false_positives['diff'].isin([1, 2, 3, 4, 5]))]['fp_index'].tolist()

print(f'True Positives: {len(true_positives)}')
print(f'Missed: {len(missed)}')
print(f'Sensitivity: {round(len(true_positives) / total_events, 3)}')
print(f'False Positives: {len(false_positives)}')